# 15. Animated GIFs: MD Trajectories, NEB Paths, Crystal Orbits & 3D Isosurfaces

Static figures are great for papers and posters, while looping **animated GIFs** bring talks, lecture slides, and lab-meeting updates to life. `xkcd-matchem` provides three one-line animation helpers—all locking shared viewports across frames so atoms, polyhedra, and orbital lobes move smoothly without zoom jitter:

- `xm.animate_trajectory()` — animates MD and NEB reaction trajectories from a `Trajectory` object or filename (with optional `ping_pong=True` forward-and-back looping).
- `xm.animate_orbit()` — sweeps a 360° camera orbit around any molecule, crystal, slab, or coordination polyhedra supercell.
- `xm.animate_isosurface()` — extracts a 3D orbital or charge-density isosurface once and orbits it in any of the 8 comic pen-and-ink shading modes (`solid`, `hatch`, `crosshatch`, `contour`, `cross_contour`, `stipple`, `scumble`, `outline`).

:::{note}
GIF export uses **Pillow** (`pip install "xkcd-matchem[animation] @ git+https://github.com/vzordillo/xkcd-matchem.git"`). Pillow is already pre-installed in Google Colab, so every cell below runs out of the box.
:::

In [ ]:
# Install xkcd-matchem automatically in Google Colab
import importlib.util


if importlib.util.find_spec("xkcd_matchem") is None:
    %pip install -q "git+https://github.com/vzordillo/xkcd-matchem.git"


## 1. Molecular Dynamics Trajectory GIF (`cu_md.lammpstrj`)

Pass either a loaded `Trajectory` or a filename directly to `xm.animate_trajectory()`. Using `IPython.display.Image(data=..., format="gif")` displays the looping GIF directly inside Jupyter, Google Colab, and Jupyter Book:

In [ ]:
import pathlib
import tempfile

from IPython.display import Image

import xkcd_matchem as xm


with tempfile.TemporaryDirectory() as tmpdir:
    gif_path = xm.animate_trajectory(
        "cu_md.lammpstrj",
        pathlib.Path(tmpdir) / "cu_md.gif",
        fps=5,
        stride=1,
        dpi=95,
        title="Cu Cluster MD",
    )
    md_gif_bytes = gif_path.read_bytes()

Image(data=md_gif_bytes, format="gif")


## 2. Reaction Pathway GIFs (`sn2_neb.xyz`)

For Climbing-Image NEB trajectories, `ping_pong=True` reverses through interior images so the transition-state inversion loops smoothly without a jarring jump back to image 1:

In [ ]:
with tempfile.TemporaryDirectory() as tmpdir:
    gif_path = xm.animate_trajectory(
        "sn2_neb.xyz",
        pathlib.Path(tmpdir) / "sn2_neb.gif",
        fps=6,
        camera=xm.Camera(azimuth=-115.0, elevation=20.0, perspective=0.25),
        ping_pong=True,
        show_cell=False,
        dpi=95,
        title="SN2 Inversion",
    )
    neb_gif_bytes = gif_path.read_bytes()

Image(data=neb_gif_bytes, format="gif")


### Custom Synchronized 2-Panel NEB Animation (3D Structure + Energy Barrier)

Need synchronized subplots—such as the 3D atomic geometry on the left paired with a live marker moving along the minimum energy path (`xm.plot_neb`) on the right? You can combine `xkcd-matchem` plotters with `PIL.Image` in ~30 lines:

In [ ]:
import io

from matplotlib import pyplot as plt
from PIL import Image as PilImage


neb = xm.load_trajectory("sn2_neb.xyz")
rel_e = neb.energy_profile(relative=True, unit="eV")
cam = xm.Camera(azimuth=-115.0, elevation=20.0, perspective=0.25)
seq = list(range(len(neb))) + list(range(len(neb) - 2, 0, -1))

frames = []
for idx in seq:
    with xm.xkcd_style("classic"):
        fig, (ax_l, ax_r) = plt.subplots(1, 2, figsize=(8.8, 3.8), dpi=95)
        xm.render_structure_3d(neb[idx], ax=ax_l, camera=cam, show_cell=False)
        ax_l.set_xlim(-3.8, 3.8)
        ax_l.set_ylim(-3.2, 3.2)
        ax_l.set_title(f"SN2 Image {idx + 1}/{len(neb)}")
        xm.plot_neb(neb, ax=ax_r, title="Minimum Energy Path")
        ax_r.scatter(
            [idx / max(len(neb) - 1, 1)],
            [float(rel_e[idx])],
            s=120,
            color="#D55E00",
            edgecolor=xm.style.INK,
            linewidth=1.5,
            zorder=10,
        )
        fig.tight_layout()
        buf = io.BytesIO()
        fig.savefig(buf, format="png", dpi=95, facecolor=fig.get_facecolor())
        plt.close(fig)
        buf.seek(0)
        frames.append(PilImage.open(buf).convert("RGB"))

synced_buf = io.BytesIO()
frames[0].save(
    synced_buf,
    format="GIF",
    save_all=True,
    append_images=frames[1:],
    duration=150,
    loop=0,
    optimize=True,
)
Image(data=synced_buf.getvalue(), format="gif")


## 3. 360° Crystal & Coordination Polyhedra Orbit (`tio2_slab.poscar`)

`xm.animate_orbit()` automatically trims excess slab vacuum when `polyhedra=` is specified, tiles supercells, locks a constant 360° viewport, and renders an inline element legend plus optional crystallographic `a, b, c` triad:

In [ ]:
with tempfile.TemporaryDirectory() as tmpdir:
    gif_path = xm.animate_orbit(
        "tio2_slab.poscar",
        pathlib.Path(tmpdir) / "tio2_orbit.gif",
        n_frames=16,
        fps=8,
        supercell=(2, 2, 1),
        polyhedra=("Ti", "O"),
        crystal_axes="corner",
        dpi=95,
        title="TiO2 (110) Polyhedra Orbit",
    )
    orbit_gif_bytes = gif_path.read_bytes()

Image(data=orbit_gif_bytes, format="gif")


## 4. 360° 3D Orbital Isosurface Orbits (`water_homo.cube`)

`xm.animate_isosurface()` extracts the Marching Tetrahedra orbital meshes **once** and orbits them cleanly across all azimuth angles using any of the 8 comic pen-and-ink shading styles. Below we render a classic white-paper **cross-hatched** orbit first, followed by a dark chalkboard (`dark_chalk`) **stippled** slide animation:

In [ ]:
with tempfile.TemporaryDirectory() as tmpdir:
    gif_path = xm.animate_isosurface(
        "water_homo.cube",
        pathlib.Path(tmpdir) / "water_homo_crosshatch.gif",
        n_frames=14,
        fps=7,
        shade="crosshatch",
        dpi=95,
        title="Water HOMO Orbit (crosshatch)",
    )
    iso_light_bytes = gif_path.read_bytes()

Image(data=iso_light_bytes, format="gif")


In [ ]:
with tempfile.TemporaryDirectory() as tmpdir:
    gif_path = xm.animate_isosurface(
        "water_homo.cube",
        pathlib.Path(tmpdir) / "water_homo_dark.gif",
        n_frames=14,
        fps=7,
        shade="stipple",
        colors=("#56B4E9", "#F0A202"),
        alpha=0.38,
        preset="dark_chalk",
        dpi=95,
        title="Water HOMO Orbit (dark_chalk + stipple)",
    )
    iso_dark_bytes = gif_path.read_bytes()

Image(data=iso_dark_bytes, format="gif")


:::{tip}
**Saving to Disk for Slides:** Replace the temporary directory path with a local filename such as `"water_homo_orbit.gif"` or `"tio2_orbit.gif"` (and increase `n_frames=24`, `dpi=120`) to save crisp looping GIFs you can drag straight into PowerPoint, Keynote, or Google Slides. Want interactive click-and-drag orbiting instead? Call `xm.view_structure("water_homo.cube", iso_shade="crosshatch")` or open the [Interactive 3D Studio](../playground.md).
:::